# Twitter Political Party Classification with BERT
This notebook is adapted to the repo dataset and loader, and organized like a practical train-and-deploy workflow.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


In [ ]:
# !pip install torch transformers scikit-learn pandas numpy fastapi uvicorn mlflow


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

from src.text_loader.loader import DataLoader


In [ ]:
MODEL_NAME = "prajjwal1/bert-tiny"   # switch to "bert-base-uncased" later
DATA_PATH = PROJECT_ROOT / "data" / "Tweets.csv"

TEST_SIZE = 0.2
VAL_SIZE = 0.1
RANDOM_STATE = 42
MAX_LENGTH = 32
BATCH_SIZE = 8
EPOCHS = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODEL_DIR = ARTIFACTS_DIR / "bert_twitter_party_model"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH =", DATA_PATH)
print("DATA EXISTS =", DATA_PATH.exists())
print("DEVICE =", DEVICE)


In [ ]:
loader = DataLoader(filepath=str(DATA_PATH))
raw_df = loader.load_data().copy()

print("Raw shape:", raw_df.shape)
raw_df.head()


In [ ]:
df = raw_df[["Tweet", "Party"]].dropna().copy()
df["text"] = df["Tweet"].astype(str).apply(loader.clean_text)
df["label_name"] = df["Party"].astype(str).apply(loader.clean_text)
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print("Cleaned shape:", df.shape)
df[["Tweet", "text", "Party", "label_name"]].head()


In [ ]:
label_names = sorted(df["label_name"].unique().tolist())
label_to_id = {label: i for i, label in enumerate(label_names)}
id_to_label = {i: label for label, i in label_to_id.items()}
df["label"] = df["label_name"].map(label_to_id)

print("label_to_id =", label_to_id)


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

val_fraction_of_train = VAL_SIZE / (1 - TEST_SIZE)

train_df, val_df = train_test_split(
    train_df,
    test_size=val_fraction_of_train,
    random_state=RANDOM_STATE,
    stratify=train_df["label"]
)

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer


In [ ]:
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=32):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


In [ ]:
train_dataset = TweetDataset(train_df["text"], train_df["label"], tokenizer, MAX_LENGTH)
val_dataset = TweetDataset(val_df["text"], val_df["label"], tokenizer, MAX_LENGTH)
test_dataset = TweetDataset(test_df["text"], test_df["label"], tokenizer, MAX_LENGTH)

sample = train_dataset[0]
for key, value in sample.items():
    print(key, value.shape if hasattr(value, "shape") else value)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }


In [ ]:
start = time.time()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id_to_label,
    label2id=label_to_id,
)

model.to(DEVICE)

print("Model loaded in", round(time.time() - start, 2), "seconds")


In [ ]:
training_args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR / "trainer_output"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    report_to="none",
    disable_tqdm=True,
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


In [ ]:
train_result = trainer.train()
train_result


In [ ]:
val_metrics = trainer.evaluate(val_dataset)
val_metrics


In [ ]:
test_output = trainer.predict(test_dataset)
test_logits = test_output.predictions
test_labels = test_output.label_ids
test_preds = np.argmax(test_logits, axis=1)

print("Test accuracy:", accuracy_score(test_labels, test_preds))
print("Test weighted F1:", f1_score(test_labels, test_preds, average="weighted"))
print()
print(classification_report(test_labels, test_preds, target_names=label_names, zero_division=0))


In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

with open(ARTIFACTS_DIR / "label_to_id.json", "w") as f:
    json.dump(label_to_id, f, indent=2)

with open(ARTIFACTS_DIR / "id_to_label.json", "w") as f:
    json.dump({str(k): v for k, v in id_to_label.items()}, f, indent=2)

print("Saved artifacts to:", ARTIFACTS_DIR)


In [ ]:
def predict_tweet(text, model, tokenizer, max_length=32):
    model.eval()
    clean_text = loader.clean_text(str(text))

    encoded = tokenizer(
        clean_text,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt",
    )

    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        pred_id = int(torch.argmax(outputs.logits, dim=1).cpu().numpy()[0])

    return id_to_label[pred_id]

example_text = "We need better public services and stronger economic policy."
print("Prediction:", predict_tweet(example_text, model, tokenizer, MAX_LENGTH))


In [ ]:
fastapi_example = '''
from fastapi import FastAPI
from pydantic import BaseModel
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ARTIFACTS_DIR = Path("artifacts")
MODEL_DIR = ARTIFACTS_DIR / "bert_twitter_party_model"

with open(ARTIFACTS_DIR / "id_to_label.json", "r") as f:
    id_to_label = {int(k): v for k, v in json.load(f).items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

app = FastAPI()

class InputText(BaseModel):
    input_texts: str

@app.get("/health")
def get_health():
    return {"status": "OK"}

@app.post("/get-prediction/")
def get_prediction(input_data: InputText):
    encoded = tokenizer(
        input_data.input_texts,
        truncation=True,
        padding="max_length",
        max_length=32,
        return_tensors="pt",
    )
    with torch.no_grad():
        outputs = model(**encoded)
        pred_id = int(torch.argmax(outputs.logits, dim=1).cpu().numpy()[0])
    return {"input": input_data.input_texts, "prediction": id_to_label[pred_id]}
'''

print(fastapi_example)


## Is this notebook design good?
Yes for an assignment or portfolio-style repo: it cleanly separates data prep, training, evaluation, artifact saving, and API handoff. The main improvement is that the original repo currently ships a TF-IDF-style loader and stub app, so this notebook adds a fuller transformer workflow on top of that.
